# Portfolio Project: Translation Time Prediction Model
This notebook implements a scratch-built **Multiple Linear Regression** pipeline to predict the translation time of a document (in minutes). 

By analyzing four core structural and semantic features_ **Word Count, Average Sentence Length, Sentence Complexity %, and Technical Terms %**_ the model automatically learns the relationship between document metrics and the labor hours required to translate them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from utils import zscore_normalize_features, gradient_descent_regression

In [ ]:
# Reading the CSV file into a Pandas DataFrame
df = pd.read_csv('translation_data.csv')

# Displaying the first few rows to ensure it loaded correctly
df.head()

In [ ]:
print("--- Dataset Information Summary ---")
print(f"Total Rows and Columns: {df.shape}\n")
print(df.describe())

# Plotting: Word Count vs Translation Time to look for trends
plt.figure(figsize=(7, 4))
plt.scatter(df['words'], df['translation_time_min'], color='darkblue', marker='o')
plt.title("Impact of Word Count on Translation Time")
plt.xlabel("Word Count")
plt.ylabel("Translation Time (Minutes)")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# Extracting the 4 feature columns as our X matrix
# pct is acronym for percentage

feature_cols = ['words', 'avg_sentence_length', 'sentence_complexity_pct', 'technical_terms_pct']
X = df[feature_cols].values

# Extract the target column as our y vector
y = df['translation_time_min'].values

# Normalize features so large word counts don't overwhelm percentage scales
X_norm, X_mu, X_sigma = zscore_normalize_features(X)

print(f"Features matrix shape: {X_norm.shape}")
print(f"Target vector shape: {y.shape}")

In [ ]:
# Initializing 4 weights (one per column) and 1 bias scalar
w_init = np.zeros(X_norm.shape[1])
b_init = 0.0

# Hyperparameters
iterations = 4000
alpha = 0.1

# Run the training loop from utils.py
w_final, b_final, J_hist = gradient_descent_regression(X_norm, y, w_init, b_init, alpha, iterations)

In [ ]:
print("=== Optimized Parameters ===")
print(f"Final Weights Vector (w_final): {w_final}")
print(f"Final Bias Scalar (b_final):    {b_final:.4f}\n")

for i, col in enumerate(feature_cols):
    print(f"Weight for '{col}': {w_final[i]:+.4f}")

In [ ]:
# Generating predictions using vectorized matrix multiplication
predictions = X_norm @ w_final + b_final

# Calculating Mean Absolute Error as our accuracy checkpoint
mae = np.mean(np.abs(predictions - y))

print(f"REGRESSION MODEL EVALUATION")
print(f"Mean Absolute Error: On average, predictions miss by {mae:.2f} minutes.\n")
print(f"{'Actual Mins':<12} | {'Predicted Mins':<15} | {'Error Margin':<12}")
print("-" * 46)

for i in range(len(y)):
    error_margin = predictions[i] - y[i]
    # \n creates an explicit line break between each row to keep text crisp
    print(f"{y[i]:<12} | {predictions[i]:<15.2f} | {error_margin:<+12.2f}\n")

# Model Evaluation & Visualization
Below, we visualize the final trained regression line against our actual training data to verify that our Gradient Descent engine successfully converged and fit the dataset trend.

In [ ]:
plt.figure(figsize=(8, 4))
# Plot the real, actual times as a scatter plot
plt.scatter(df['words'], y, color='blue', label='Actual Data', alpha=0.7)

# Sort values to ensure the regression line plots smoothly across the graph
sort_idx = np.argsort(df['words'])
plt.plot(df['words'].iloc[sort_idx], predictions[sort_idx], color='red', linewidth=2, label='Regression Fit')

plt.title("Model Regression Line vs Actual Data")
plt.xlabel("Word Count")
plt.ylabel("Translation Time (Minutes)")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()